In [2]:
from torch import nn
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

## 1. Getting a datasett

In [3]:
train_data = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=None   # In what format the labels/target we want to transform
)

In [4]:
test_data = datasets.FashionMNIST(
    root='data',
    train=False,
    download=True,
    transform=ToTensor(),
    target_transform=None
)

In [5]:
len(train_data)

60000

In [6]:
train_data

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [8]:
class_names = train_data.classes
class_names

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [9]:
class_to_idx = train_data.class_to_idx
class_to_idx

{'T-shirt/top': 0,
 'Trouser': 1,
 'Pullover': 2,
 'Dress': 3,
 'Coat': 4,
 'Sandal': 5,
 'Shirt': 6,
 'Sneaker': 7,
 'Bag': 8,
 'Ankle boot': 9}

In [10]:
train_data.targets

tensor([9, 0, 0,  ..., 3, 0, 5])

## 2. Creating a dataloader

In [12]:
from torch.utils.data import DataLoader

batch_size  = 32

train_dataloader = DataLoader(dataset=train_data,
                              batch_size=32,
                              shuffle=True)

test_dataloader = DataLoader(dataset=test_data,
                              batch_size=32,
                              shuffle=True)

train_dataloader , test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x1dc330d9d30>,
 <torch.utils.data.dataloader.DataLoader at 0x1dc3309d310>)

In [13]:
train_features_batch , train_label_batch = next(iter(train_dataloader))

## 3. Creating a baseline model

In [14]:
trian_data = train_features_batch[3]
trian_data

tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0078, 0.0196, 0.0000, 0.0000, 0.3216, 0.4863, 0.3451, 0.3333,
          0.4980, 0.0118, 0.0000, 0.0078, 0.0196, 0.0078, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0118,
          0.0000, 0.0000, 0.0000, 0.0000, 0.6588, 0.6784, 0.8431, 0.7647,
          0.6392, 0.2510, 0.0000, 0.0000, 0.0000, 0.0000, 0.0118, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000,
          0.0078, 0.3216, 0.5882, 0.6392, 0.6863, 0.6314, 0.7098, 0.7490,
          0.6235, 0.6902, 0.6039, 0.5451, 0.2314, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.3961,
          0.5961, 0.6353, 0.6314, 0.5961, 0.6118, 0.7176, 0.6196, 0.5882,
          0.6039, 0.5804, 0.5961, 0.6235, 0.6510, 0.5882,

In [15]:
from torch import nn

class fashionmodel(nn.Module):
    def __init__(self, 
                 input_shape,
                 hidden_unit,
                 output_shape):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_shape,
                      out_features=hidden_unit),
            nn.Linear(in_features=hidden_unit,
                      out_features=output_shape)
        )

    def forward(self , x):
        return self.layer_stack(x)

In [24]:
import torch
torch.manual_seed(42)

model_0 = fashionmodel(784 , 10 , len(class_to_idx))
model_0

fashionmodel(
  (layer_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=10, bias=True)
    (2): Linear(in_features=10, out_features=10, bias=True)
  )
)

In [25]:
loss_fun = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(params=model_0.parameters(),
                            lr=0.1)

In [26]:
# Creating a function to time our experiment

from timeit import default_timer as timer

def print_train_time(start , 
                     end):
    total_time = end - start
    print(f"The total time is : {total_time : .3f} sec")
    return total_time


In [27]:
start_time = timer()

end_time = timer()
print_train_time(start_time ,end_time)

The total time is :  0.000 sec


3.819999983534217e-05

## 4. Building a training loop

In [31]:
from tqdm.auto import tqdm
import torch
from torch import nn

# Accuracy calculation function
def accuracy_fn(y_true, y_pred):
    """Calculate accuracy between truth labels and predictions"""
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

# Training loop with accuracy tracking
torch.manual_seed(42)
train_time_start = timer()

epochs = 3

for epoch in tqdm(range(epochs)):
    print(f"Epoch : {epoch}\n--------")

    ### Training ###
    train_loss = 0
    train_acc = 0
    
    model_0.train()
    for batch, (image, label) in enumerate(train_dataloader):
        # Forward pass
        y_pred = model_0(image)
        
        # Calculate loss
        loss = loss_fun(y_pred, label)
        train_loss += loss.item()
        
        # Calculate accuracy
        y_pred_class = torch.argmax(y_pred, dim=1)
        train_acc += accuracy_fn(label, y_pred_class)
        
        # Optimizer zero grad
        optimiser.zero_grad()
        
        # Backward pass
        loss.backward()
        
        # Optimizer step
        optimiser.step()

        if batch % 400 == 0:
            print(f'Looked at {batch * len(image)}/{len(train_dataloader.dataset)} samples.')
    
    # Average loss and accuracy per batch
    train_loss /= len(train_dataloader)
    train_acc /= len(train_dataloader)

    ### Testing ###
    test_loss = 0
    test_acc = 0
    
    model_0.eval()
    with torch.inference_mode():
        for image_test, label_test in test_dataloader:
            # Forward pass
            test_pred = model_0(image_test)
            
            # Calculate loss
            test_loss += loss_fun(test_pred, label_test).item()
            
            # Calculate accuracy
            test_pred_class = torch.argmax(test_pred, dim=1)
            test_acc += accuracy_fn(label_test, test_pred_class)
        
        # Average loss and accuracy per batch
        test_loss /= len(test_dataloader)
        test_acc /= len(test_dataloader)

    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%\n')

train_time_end = timer()
total_train_time = print_train_time(train_time_start, train_time_end)

# Print final results
print(f"\n{'='*50}")
print(f"Final Results after {epochs} epochs:")
print(f"{'='*50}")
print(f"Training Time: {total_train_time:.3f} seconds")
print(f"Final Train Accuracy: {train_acc:.2f}%")
print(f"Final Test Accuracy: {test_acc:.2f}%")
print(f"{'='*50}")


  0%|          | 0/3 [00:00<?, ?it/s]

Epoch : 0
--------
Looked at 0/60000 samples.
Looked at 12800/60000 samples.
Looked at 25600/60000 samples.
Looked at 38400/60000 samples.
Looked at 51200/60000 samples.
Train Loss: 0.4363 | Train Acc: 84.71%
Test Loss: 0.4618 | Test Acc: 83.71%

Epoch : 1
--------
Looked at 0/60000 samples.
Looked at 12800/60000 samples.
Looked at 25600/60000 samples.
Looked at 38400/60000 samples.
Looked at 51200/60000 samples.
Train Loss: 0.4287 | Train Acc: 85.06%
Test Loss: 0.4966 | Test Acc: 82.94%

Epoch : 2
--------
Looked at 0/60000 samples.
Looked at 12800/60000 samples.
Looked at 25600/60000 samples.
Looked at 38400/60000 samples.
Looked at 51200/60000 samples.
Train Loss: 0.4250 | Train Acc: 85.20%
Test Loss: 0.5076 | Test Acc: 83.11%

The total time is :  19.225 sec

Final Results after 3 epochs:
Training Time: 19.225 seconds
Final Train Accuracy: 85.20%
Final Test Accuracy: 83.11%


In [23]:
image.shape , label.shape

(torch.Size([32, 1, 28, 28]), torch.Size([32]))